# Lab 2: Modelos e Deployment (20 min)

## Objetivos
- Entender o catálogo de modelos do Foundry
- Usar o Playground para testar modelos
- Ver métricas de um deployment
- Consumir modelos via código Python

## 📖 Conceitos

### Tipos de Deployment

| Tipo | Descrição | Quando usar |
|------|-----------|-------------|
| **Global Standard** | Partilhado, pay-per-token | Testes, baixo volume |
| **Standard** | Dedicado por região | Produção, latência previsível |
| **Provisioned** | Capacidade reservada (PTU) | Alto volume, SLA garantido |

### Modelos Populares

| Modelo | Uso | Tokens/Contexto |
|--------|-----|------------------|
| GPT-4o | Chat, raciocínio, visão | 128K |
| GPT-4o-mini | Chat rápido e barato | 128K |
| o3-mini | Raciocínio avançado | 200K |
| text-embedding-ada-002 | Embeddings para search | 8K |

## 🖥️ Exercício 2.1: Playground (Portal)

1. Vai ao [Portal Foundry](https://ai.azure.com) → **Playground**
2. Seleciona o teu deployment `gpt-4o`
3. No **System Message** escreve:
   ```
   Tu és um assistente simpático que responde em português de Portugal.
   Sê conciso e usa emojis.
   ```
4. Testa estas perguntas:
   - "O que é machine learning?"
   - "Explica o que é uma API como se eu tivesse 10 anos"
   - "Escreve um haiku sobre código"

5. Experimenta alterar o **Temperature** (0 vs 1) e ver a diferença nas respostas

> 💡 **Dica:** Temperature baixa = respostas mais determinísticas. Alta = mais criativas.

## 🖥️ Exercício 2.2: Ver Métricas do Deployment (Portal)

1. No portal, vai a **Deployments** → seleciona o teu deployment `gpt-4o`
2. Observa:
   - **Requests**: quantos pedidos foram feitos
   - **Token Usage**: quantos tokens foram consumidos
   - **Latency**: tempo de resposta
   - **Rate Limits**: limites do teu deployment

## 💻 Exercício 2.3: Consumir o Modelo via Código

Agora vamos fazer o mesmo que fizemos no Playground, mas com Python.

In [ ]:
# Setup - carregar configuração
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
print(f"✅ Cliente configurado | Deployment: {DEPLOYMENT}")

In [ ]:
# 2.3a - Chat simples
response = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[
        {
            "role": "system",
            "content": "Tu és um assistente simpático que responde em português de Portugal. Sê conciso e usa emojis.",
        },
        {
            "role": "user",
            "content": "O que é machine learning?",
        },
    ],
    temperature=0.7,
    max_tokens=200,
)

print("🤖 Resposta:")
print(response.choices[0].message.content)
print(f"\n📊 Tokens: {response.usage.prompt_tokens} (prompt) + {response.usage.completion_tokens} (resposta) = {response.usage.total_tokens} (total)")

In [ ]:
# 2.3b - Conversa multi-turno (com histórico)
mensagens = [
    {"role": "system", "content": "Tu és um professor de programação. Explica conceitos de forma simples."},
    {"role": "user", "content": "O que é uma variável?"},
]

# Primeira resposta
r1 = client.chat.completions.create(model=DEPLOYMENT, messages=mensagens, max_tokens=150)
resposta1 = r1.choices[0].message.content
print("👤 O que é uma variável?")
print(f"🤖 {resposta1}")

# Adicionar ao histórico e fazer follow-up
mensagens.append({"role": "assistant", "content": resposta1})
mensagens.append({"role": "user", "content": "Podes dar um exemplo em Python?"})

r2 = client.chat.completions.create(model=DEPLOYMENT, messages=mensagens, max_tokens=200)
print(f"\n👤 Podes dar um exemplo em Python?")
print(f"🤖 {r2.choices[0].message.content}")

In [ ]:
# 2.3c - Streaming (resposta aparece token a token)
print("🤖 Resposta em streaming:")
print("-" * 40)

stream = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[
        {"role": "user", "content": "Conta-me uma história curta (3 frases) sobre um robot que aprende a cozinhar."},
    ],
    max_tokens=200,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n" + "-" * 40)
print("✅ Stream completo!")

In [ ]:
# 2.3d - Comparar temperaturas
pergunta = "Inventa um nome criativo para uma startup de IA."

for temp in [0.0, 0.5, 1.0]:
    print(f"\n🌡️ Temperature = {temp}")
    for i in range(3):
        r = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[{"role": "user", "content": pergunta}],
            temperature=temp,
            max_tokens=20,
        )
        print(f"   Tentativa {i+1}: {r.choices[0].message.content.strip()}")

## 🧪 Desafio Extra (se tiveres tempo)

Tenta criar um system prompt que faça o modelo responder sempre em formato JSON:

```python
# Dica: usa response_format
response = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[...],
    response_format={"type": "json_object"},
)
```

## ✅ Resumo

Neste lab aprendeste:
- Os tipos de deployment disponíveis
- Como usar o Playground para testar modelos
- Como ver métricas do deployment
- Como consumir modelos via Python (chat simples, multi-turno, streaming)
- O efeito da temperatura nas respostas

**Próximo:** [Lab 3 - Agentes](lab03-agentes.ipynb)